In [13]:
import os
import numpy as np
import pandas as pd

# Competition data path
BASE = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

TRAIN_DIR = os.path.join(BASE, "train")
TEST_DIR = os.path.join(BASE, "test")

# Load CSV files
train = pd.read_csv(os.path.join(BASE, "train.csv"))
test = pd.read_csv(os.path.join(BASE, "test.csv"))
sample = pd.read_csv(os.path.join(BASE, "sample_submission.csv"))

print("Dataset loaded successfully!")
print("Train:", train.shape)
print("Test :", test.shape)
print("Sample:", sample.shape)

print("\nTrain columns:", train.columns.tolist())
print("Test columns :", test.columns.tolist())
print("Sample columns:", sample.columns.tolist())

Dataset loaded successfully!
Train: (769, 2)
Test : (216, 2)
Sample: (204, 2)

Train columns: ['filename', 'label']
Test columns : ['filename', 'label']
Sample columns: ['filename', 'label']


In [14]:
# Basic data inspection

print("TRAIN DATA")
display(train.head())

print("\nTraining label statistics:")
display(train["label"].describe())

print("\nTraining label values:")
print(sorted(train["label"].unique()))

print("\nMissing values:")
print(train.isnull().sum())

print("\nAudio files:")
print("Train audio files:", len(os.listdir(TRAIN_DIR)))
print("Test audio files :", len(os.listdir(TEST_DIR)))

TRAIN DATA


,filename,label
0,audio_192.wav,3.0
1,audio_436.wav,3.0
2,audio_447.wav,3.5
3,audio_126.wav,2.0
4,audio_61.wav,2.0



Training label statistics:


count    769.000000
mean       3.313394
std        1.239000
min        0.000000
25%        2.500000
50%        3.000000
75%        4.000000
max        5.000000
Name: label, dtype: float64


Training label values:
[np.float64(0.0), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]

Missing values:
filename    0
label       0
dtype: int64

Audio files:
Train audio files: 769
Test audio files : 216


In [15]:
# Audio feature extraction

import librosa
from tqdm.auto import tqdm

def extract_audio_features(filepath):
    y, sr = librosa.load(filepath, sr=16000, mono=True)

    features = {}

    # -------------------------
    # 1. Duration
    # -------------------------
    features["duration"] = len(y) / sr

    # -------------------------
    # 2. Energy
    # -------------------------
    rms = librosa.feature.rms(y=y)[0]

    features["rms_mean"] = np.mean(rms)
    features["rms_std"] = np.std(rms)
    features["rms_max"] = np.max(rms)

    # -------------------------
    # 3. Zero Crossing Rate
    # -------------------------
    zcr = librosa.feature.zero_crossing_rate(y)[0]

    features["zcr_mean"] = np.mean(zcr)
    features["zcr_std"] = np.std(zcr)

    # -------------------------
    # 4. Spectral features
    # -------------------------
    spectral_centroid = librosa.feature.spectral_centroid(
        y=y, sr=sr
    )[0]

    spectral_bandwidth = librosa.feature.spectral_bandwidth(
        y=y, sr=sr
    )[0]

    spectral_rolloff = librosa.feature.spectral_rolloff(
        y=y, sr=sr
    )[0]

    features["centroid_mean"] = np.mean(spectral_centroid)
    features["centroid_std"] = np.std(spectral_centroid)

    features["bandwidth_mean"] = np.mean(spectral_bandwidth)
    features["bandwidth_std"] = np.std(spectral_bandwidth)

    features["rolloff_mean"] = np.mean(spectral_rolloff)
    features["rolloff_std"] = np.std(spectral_rolloff)

    # -------------------------
    # 5. MFCC features
    # -------------------------
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=13
    )

    for i in range(13):
        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
        features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])

    return features


print("Feature extraction function created successfully!")

Feature extraction function created successfully!


In [16]:
# Extract features from all training audio files

train_features = []

for filename in tqdm(train["filename"], desc="Processing training audio"):
    filepath = os.path.join(TRAIN_DIR, filename)

    try:
        features = extract_audio_features(filepath)
        features["filename"] = filename
        train_features.append(features)

    except Exception as e:
        print(f"Error processing {filename}: {e}")

train_features = pd.DataFrame(train_features)

print("\nTraining feature extraction complete!")
print("Feature matrix shape:", train_features.shape)

display(train_features.head())

Processing training audio:   0%|          | 0/769 [00:00<?, ?it/s]


Training feature extraction complete!
Feature matrix shape: (769, 39)


,duration,rms_mean,rms_std,rms_max,zcr_mean,zcr_std,centroid_mean,centroid_std,bandwidth_mean,bandwidth_std,...,mfcc_9_std,mfcc_10_mean,mfcc_10_std,mfcc_11_mean,mfcc_11_std,mfcc_12_mean,mfcc_12_std,mfcc_13_mean,mfcc_13_std,filename
0,60.508375,0.031786,0.038404,0.188842,0.084768,0.053518,979.478539,399.567888,916.472067,261.620566,...,9.175092,-6.111350,7.005802,-11.947043,8.755229,-8.226156,6.883492,-6.238205,5.747041,audio_192.wav
1,60.074688,0.068651,0.055423,0.400792,0.142977,0.127519,1813.837012,1156.460316,1502.301143,495.435560,...,16.002325,-8.470199,13.448154,-9.261223,11.554407,-9.040758,13.295259,4.751498,13.532232,audio_436.wav
2,60.074688,0.054421,0.051533,0.249218,0.146870,0.129777,1652.475254,1207.838332,1500.652559,535.214385,...,16.530882,-8.570185,16.668097,3.306296,13.158398,-3.113312,11.855078,-0.866495,11.763124,audio_447.wav
3,45.060000,0.013778,0.012183,0.061771,0.059477,0.036870,996.631363,471.566967,1177.252185,392.434373,...,15.149113,-21.515909,15.097207,-12.347360,11.821060,-9.993058,10.599061,-17.710949,11.394984,audio_126.wav
4,45.300000,0.009330,0.008970,0.042700,0.169988,0.128430,2278.144095,1068.412091,1630.415181,442.415960,...,16.179859,-18.795961,16.087769,-8.590364,13.599857,3.873606,10.692766,-17.764479,15.302751,audio_61.wav


In [17]:
# Extract features from all test audio files

test_features = []

for filename in tqdm(test["filename"], desc="Processing test audio"):
    filepath = os.path.join(TEST_DIR, filename)

    try:
        features = extract_audio_features(filepath)
        features["filename"] = filename
        test_features.append(features)

    except Exception as e:
        print(f"Error processing {filename}: {e}")

test_features = pd.DataFrame(test_features)

print("\nTest feature extraction complete!")
print("Feature matrix shape:", test_features.shape)

display(test_features.head())

Processing test audio:   0%|          | 0/216 [00:00<?, ?it/s]


Test feature extraction complete!
Feature matrix shape: (216, 39)


,duration,rms_mean,rms_std,rms_max,zcr_mean,zcr_std,centroid_mean,centroid_std,bandwidth_mean,bandwidth_std,...,mfcc_9_std,mfcc_10_mean,mfcc_10_std,mfcc_11_mean,mfcc_11_std,mfcc_12_mean,mfcc_12_std,mfcc_13_mean,mfcc_13_std,filename
0,45.060000,0.018476,0.018033,0.096424,0.091145,0.061664,1313.264772,629.115488,1370.977849,523.550599,...,10.750962,-8.662834,11.958874,-2.093964,8.446618,-2.751901,8.443857,-5.733756,8.805126,audio_128.wav
1,44.969313,0.046379,0.045599,0.219015,0.186478,0.134429,1887.407841,1131.228708,1602.602286,499.222545,...,17.222078,-18.275993,13.206761,-6.153337,13.933970,-5.587032,10.239784,-7.673638,9.794288,audio_156.wav
2,45.012000,0.048284,0.044188,0.233504,0.162025,0.082149,1677.448618,724.928917,1313.941010,388.693842,...,13.497080,-10.828346,12.366036,-17.725065,11.668586,-2.528805,10.497900,-7.882828,9.199354,audio_19.wav
3,58.980000,0.005259,0.005214,0.029752,0.159751,0.109905,1915.431594,861.140927,1736.017440,322.650005,...,9.221682,-1.623224,10.203712,-9.178815,7.352479,-6.522888,6.525156,-6.726883,7.192762,audio_108.wav
4,45.040000,0.058809,0.038030,0.209144,0.064533,0.065902,1171.666526,725.126069,1447.879524,452.048982,...,14.190732,-0.635752,11.704207,4.300982,9.297211,-1.969271,11.009748,1.723401,9.557956,audio_206.wav


In [18]:
# Prepare data for machine learning

from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

# Separate features and target
X = train_features.drop(columns=["filename"])
y = train["label"].values

# Make sure training and feature rows have the same order
assert len(X) == len(y)

# 5-fold cross-validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Extra Trees regression model
model = make_pipeline(
    SimpleImputer(strategy="median"),
    ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        max_features=0.8,
        min_samples_leaf=2
    )
)

# Evaluate using RMSE
scores = cross_val_score(
    model,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=1
)

rmse_scores = -scores

print("5-Fold RMSE:")
for i, score in enumerate(rmse_scores, 1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMean RMSE: {rmse_scores.mean():.4f}")
print(f"Std RMSE : {rmse_scores.std():.4f}")

5-Fold RMSE:
Fold 1: 0.6910
Fold 2: 0.7391
Fold 3: 0.8076
Fold 4: 0.7571
Fold 5: 0.8338

Mean RMSE: 0.7657
Std RMSE : 0.0505


In [19]:
# Train the baseline model on all training data
# and predict the 216 test recordings

X_train = train_features.drop(columns=["filename"])
y_train = train["label"].values

X_test = test_features.drop(columns=["filename"])

# Train on all available training data
model.fit(X_train, y_train)

# Predict test scores
test_predictions = model.predict(X_test)

# Keep predictions within the valid 0–5 grammar-score range
test_predictions = np.clip(test_predictions, 0, 5)

print("Predictions generated:", len(test_predictions))
print("Minimum prediction:", test_predictions.min())
print("Maximum prediction:", test_predictions.max())
print("Mean prediction:", test_predictions.mean())

Predictions generated: 216
Minimum prediction: 2.53
Maximum prediction: 4.283055555555556
Mean prediction: 3.2440946502057613


In [20]:
# Create submission file

submission = pd.DataFrame({
    "filename": test["filename"],
    "label": test_predictions
})

# Save submission
submission_path = "/kaggle/working/submission_baseline.csv"
submission.to_csv(submission_path, index=False)

print("Submission created!")
print("Shape:", submission.shape)
print("Saved to:", submission_path)

display(submission.head(10))

Submission created!
Shape: (216, 2)
Saved to: /kaggle/working/submission_baseline.csv


,filename,label
0,audio_128.wav,3.232222
1,audio_156.wav,3.952500
2,audio_19.wav,3.298333
3,audio_108.wav,2.605556
4,audio_206.wav,3.256389
5,audio_161.wav,2.889167
6,audio_215.wav,3.001667
7,audio_213.wav,3.003611
8,audio_11.wav,3.308333
9,audio_155.wav,3.343889


In [21]:
# Final submission checks

print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())

print("\nMissing values:")
print(submission.isnull().sum())

print("\nDuplicate filenames:")
print(submission["filename"].duplicated().sum())

print("\nFilename order matches test:",
      submission["filename"].tolist() == test["filename"].tolist())

print("\nPrediction range:")
print(submission["label"].min(), "to", submission["label"].max())

Rows: 216
Columns: ['filename', 'label']

Missing values:
filename    0
label       0
dtype: int64

Duplicate filenames:
0

Filename order matches test: True

Prediction range:
2.53 to 4.283055555555556


In [22]:
# Check whether files appearing in both train and test are actually identical

import hashlib

def file_hash(filepath):
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

common_files = sorted(set(train["filename"]) & set(test["filename"]))

print("Common filenames:", len(common_files))

same_files = []
different_files = []

for filename in tqdm(common_files, desc="Comparing common audio files"):
    train_file = os.path.join(TRAIN_DIR, filename)
    test_file = os.path.join(TEST_DIR, filename)

    train_hash = file_hash(train_file)
    test_hash = file_hash(test_file)

    if train_hash == test_hash:
        same_files.append(filename)
    else:
        different_files.append(filename)

print("\nIdentical audio files:", len(same_files))
print("Different audio files :", len(different_files))

print("\nFirst identical files:")
print(same_files[:20])

print("\nFirst different files:")
print(different_files[:20])

Common filenames: 212


Comparing common audio files:   0%|          | 0/212 [00:00<?, ?it/s]


Identical audio files: 0
Different audio files : 212

First identical files:
[]

First different files:
['audio_0.wav', 'audio_1.wav', 'audio_10.wav', 'audio_100.wav', 'audio_101.wav', 'audio_102.wav', 'audio_103.wav', 'audio_104.wav', 'audio_105.wav', 'audio_106.wav', 'audio_107.wav', 'audio_108.wav', 'audio_109.wav', 'audio_11.wav', 'audio_110.wav', 'audio_111.wav', 'audio_112.wav', 'audio_113.wav', 'audio_114.wav', 'audio_115.wav']


In [24]:
# Compare several regression models using the same audio features

from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

X = train_features.drop(columns=["filename"])
y = train["label"].values

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=400,
        random_state=42,
        n_jobs=-1,
        max_features=0.8,
        min_samples_leaf=2
    ),

    "RandomForest": RandomForestRegressor(
        n_estimators=400,
        random_state=42,
        n_jobs=-1,
        max_features=0.8,
        min_samples_leaf=2
    ),

    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=2,
        loss="huber",
        random_state=42
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.04,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=42
    )
}

results = {}

for name, estimator in models.items():

    pipe = make_pipeline(
        SimpleImputer(strategy="median"),
        estimator
    )

    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=kf,
        scoring="neg_root_mean_squared_error",
        n_jobs=1
    )

    rmse = -scores.mean()
    results[name] = rmse

    print(f"{name:20s} RMSE: {rmse:.4f}")

print("\nBest model:")
best_model_name = min(results, key=results.get)
print(best_model_name, "->", f"{results[best_model_name]:.4f}")

ExtraTrees           RMSE: 0.7638
RandomForest         RMSE: 0.7876
GradientBoosting     RMSE: 0.8269
HistGradientBoosting RMSE: 0.7882

Best model:
ExtraTrees -> 0.7638


In [26]:
# Enhanced audio feature extraction
# Adds pitch-related and spectral-contrast features

def extract_enhanced_features(filepath):
    y, sr = librosa.load(filepath, sr=16000, mono=True)

    features = {}

    # -------------------------
    # Basic
    # -------------------------
    features["duration"] = len(y) / sr

    # -------------------------
    # Energy
    # -------------------------
    rms = librosa.feature.rms(y=y)[0]

    features["rms_mean"] = np.mean(rms)
    features["rms_std"] = np.std(rms)
    features["rms_min"] = np.min(rms)
    features["rms_max"] = np.max(rms)

    # -------------------------
    # Zero Crossing Rate
    # -------------------------
    zcr = librosa.feature.zero_crossing_rate(y)[0]

    features["zcr_mean"] = np.mean(zcr)
    features["zcr_std"] = np.std(zcr)

    # -------------------------
    # Spectral features
    # -------------------------
    centroid = librosa.feature.spectral_centroid(
        y=y, sr=sr
    )[0]

    bandwidth = librosa.feature.spectral_bandwidth(
        y=y, sr=sr
    )[0]

    rolloff = librosa.feature.spectral_rolloff(
        y=y, sr=sr
    )[0]

    features["centroid_mean"] = np.mean(centroid)
    features["centroid_std"] = np.std(centroid)

    features["bandwidth_mean"] = np.mean(bandwidth)
    features["bandwidth_std"] = np.std(bandwidth)

    features["rolloff_mean"] = np.mean(rolloff)
    features["rolloff_std"] = np.std(rolloff)

    # -------------------------
    # Spectral contrast
    # -------------------------
    contrast = librosa.feature.spectral_contrast(
        y=y,
        sr=sr
    )

    for i in range(contrast.shape[0]):
        features[f"contrast_{i+1}_mean"] = np.mean(contrast[i])
        features[f"contrast_{i+1}_std"] = np.std(contrast[i])

    # -------------------------
    # MFCC
    # -------------------------
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=20
    )

    for i in range(20):
        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
        features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])

    # -------------------------
    # Pitch / fundamental frequency
    # -------------------------
    f0 = librosa.yin(
        y,
        fmin=70,
        fmax=400,
        sr=sr
    )

    # Remove invalid values
    f0_valid = f0[np.isfinite(f0)]

    if len(f0_valid) > 0:
        features["pitch_mean"] = np.mean(f0_valid)
        features["pitch_std"] = np.std(f0_valid)
        features["pitch_min"] = np.min(f0_valid)
        features["pitch_max"] = np.max(f0_valid)
    else:
        features["pitch_mean"] = 0
        features["pitch_std"] = 0
        features["pitch_min"] = 0
        features["pitch_max"] = 0

    return features


print("Enhanced feature function created successfully!")

Enhanced feature function created successfully!


In [27]:
# Extract enhanced features from all training audio files

enhanced_train_features = []

for filename in tqdm(
    train["filename"],
    desc="Processing enhanced training audio"
):
    filepath = os.path.join(TRAIN_DIR, filename)

    try:
        features = extract_enhanced_features(filepath)
        features["filename"] = filename
        enhanced_train_features.append(features)

    except Exception as e:
        print(f"Error processing {filename}: {e}")

enhanced_train_features = pd.DataFrame(enhanced_train_features)

print("\nEnhanced training feature extraction complete!")
print("Feature matrix shape:", enhanced_train_features.shape)

display(enhanced_train_features.head())

Processing enhanced training audio:   0%|          | 0/769 [00:00<?, ?it/s]


Enhanced training feature extraction complete!
Feature matrix shape: (769, 72)


,duration,rms_mean,rms_std,rms_min,rms_max,zcr_mean,zcr_std,centroid_mean,centroid_std,bandwidth_mean,...,mfcc_18_std,mfcc_19_mean,mfcc_19_std,mfcc_20_mean,mfcc_20_std,pitch_mean,pitch_std,pitch_min,pitch_max,filename
0,60.508375,0.031786,0.038404,0.000104,0.188842,0.084768,0.053518,979.478539,399.567888,916.472067,...,5.152612,-8.177850,5.367302,-6.500931,5.128804,161.465261,46.958187,69.868996,400.0,audio_192.wav
1,60.074688,0.068651,0.055423,0.000282,0.400792,0.142977,0.127519,1813.837012,1156.460316,1502.301143,...,10.548928,-6.763613,11.536690,-3.490100,13.783540,222.000166,62.143378,69.868996,400.0,audio_436.wav
2,60.074688,0.054421,0.051533,0.000957,0.249218,0.146870,0.129777,1652.475254,1207.838332,1500.652559,...,9.223493,-1.456384,7.927880,-1.399758,7.972776,163.676122,73.596035,69.868996,400.0,audio_447.wav
3,45.060000,0.013778,0.012183,0.000037,0.061771,0.059477,0.036870,996.631363,471.566967,1177.252185,...,7.599261,-7.995685,7.307787,-3.190723,6.961601,168.193519,74.423537,75.871843,400.0,audio_126.wav
4,45.300000,0.009330,0.008970,0.000003,0.042700,0.169988,0.128430,2278.144095,1068.412091,1630.415181,...,8.370968,-3.073984,7.661169,-2.159748,5.829678,155.022480,78.265911,69.868996,400.0,audio_61.wav


In [28]:
# Extract enhanced features from all test audio files

enhanced_test_features = []

for filename in tqdm(
    test["filename"],
    desc="Processing enhanced test audio"
):
    filepath = os.path.join(TEST_DIR, filename)

    try:
        features = extract_enhanced_features(filepath)
        features["filename"] = filename
        enhanced_test_features.append(features)

    except Exception as e:
        print(f"Error processing {filename}: {e}")

enhanced_test_features = pd.DataFrame(enhanced_test_features)

print("\nEnhanced test feature extraction complete!")
print("Feature matrix shape:", enhanced_test_features.shape)

display(enhanced_test_features.head())

Processing enhanced test audio:   0%|          | 0/216 [00:00<?, ?it/s]


Enhanced test feature extraction complete!
Feature matrix shape: (216, 72)


,duration,rms_mean,rms_std,rms_min,rms_max,zcr_mean,zcr_std,centroid_mean,centroid_std,bandwidth_mean,...,mfcc_18_std,mfcc_19_mean,mfcc_19_std,mfcc_20_mean,mfcc_20_std,pitch_mean,pitch_std,pitch_min,pitch_max,filename
0,45.060000,0.018476,0.018033,0.000266,0.096424,0.091145,0.061664,1313.264772,629.115488,1370.977849,...,5.944876,-1.053733,5.838058,-3.126958,5.694934,143.629097,66.019019,69.868996,400.0,audio_128.wav
1,44.969313,0.046379,0.045599,0.000597,0.219015,0.186478,0.134429,1887.407841,1131.228708,1602.602286,...,7.858410,-13.392183,8.068741,-8.865784,6.943868,181.842648,60.815081,69.868996,400.0,audio_156.wav
2,45.012000,0.048284,0.044188,0.000504,0.233504,0.162025,0.082149,1677.448618,724.928917,1313.941010,...,7.054818,-8.604047,6.603037,-5.421148,6.687274,165.582067,53.866085,69.868996,400.0,audio_19.wav
3,58.980000,0.005259,0.005214,0.000203,0.029752,0.159751,0.109905,1915.431594,861.140927,1736.017440,...,5.858552,-6.269709,4.886087,-7.348907,4.604700,155.067581,49.772029,69.868996,400.0,audio_108.wav
4,45.040000,0.058809,0.038030,0.000000,0.209144,0.064533,0.065902,1171.666526,725.126069,1447.879524,...,6.675319,-6.266178,7.768198,-0.125192,6.491553,123.400675,37.181162,69.868996,400.0,audio_206.wav


In [29]:
# Validate ExtraTrees using enhanced 72-feature set

from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

X_enhanced = enhanced_train_features.drop(columns=["filename"])
y = train["label"].values

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

enhanced_model = make_pipeline(
    SimpleImputer(strategy="median"),
    ExtraTreesRegressor(
        n_estimators=400,
        random_state=42,
        n_jobs=-1,
        max_features=0.8,
        min_samples_leaf=2
    )
)

scores = cross_val_score(
    enhanced_model,
    X_enhanced,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=1
)

rmse_scores = -scores

print("Enhanced ExtraTrees CV Results")
print("--------------------------------")

for i, score in enumerate(rmse_scores, 1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMean RMSE: {rmse_scores.mean():.4f}")
print(f"Std RMSE : {rmse_scores.std():.4f}")

Enhanced ExtraTrees CV Results
--------------------------------
Fold 1: 0.6858
Fold 2: 0.7034
Fold 3: 0.7903
Fold 4: 0.7250
Fold 5: 0.8140

Mean RMSE: 0.7437
Std RMSE : 0.0499


In [30]:
# Train enhanced ExtraTrees model on the full training set

X_train_enhanced = enhanced_train_features.drop(columns=["filename"])
y_train = train["label"].values

X_test_enhanced = enhanced_test_features.drop(columns=["filename"])

enhanced_model.fit(
    X_train_enhanced,
    y_train
)

enhanced_predictions = enhanced_model.predict(X_test_enhanced)

# Keep predictions within the valid score range
enhanced_predictions = np.clip(enhanced_predictions, 0, 5)

print("Enhanced model trained successfully!")
print("Number of predictions:", len(enhanced_predictions))
print("Minimum prediction:", enhanced_predictions.min())
print("Maximum prediction:", enhanced_predictions.max())
print("Mean prediction:", enhanced_predictions.mean())

Enhanced model trained successfully!
Number of predictions: 216
Minimum prediction: 2.4387499999999998
Maximum prediction: 4.404166666666666
Mean prediction: 3.2581394675925925


In [31]:
# Create Submission #2 using enhanced features

submission_enhanced = pd.DataFrame({
    "filename": test["filename"],
    "label": enhanced_predictions
})

enhanced_submission_path = "/kaggle/working/submission_enhanced.csv"

submission_enhanced.to_csv(
    enhanced_submission_path,
    index=False
)

print("Enhanced submission created!")
print("Shape:", submission_enhanced.shape)
print("Saved to:", enhanced_submission_path)

print("\nValidation:")
print("Columns:", submission_enhanced.columns.tolist())
print("Missing values:")
print(submission_enhanced.isnull().sum())
print("Duplicate filenames:", submission_enhanced["filename"].duplicated().sum())
print(
    "Filename order matches test:",
    submission_enhanced["filename"].equals(test["filename"])
)

display(submission_enhanced.head(10))

Enhanced submission created!
Shape: (216, 2)
Saved to: /kaggle/working/submission_enhanced.csv

Validation:
Columns: ['filename', 'label']
Missing values:
filename    0
label       0
dtype: int64
Duplicate filenames: 0
Filename order matches test: True


,filename,label
0,audio_128.wav,3.310833
1,audio_156.wav,3.959583
2,audio_19.wav,3.479167
3,audio_108.wav,2.671458
4,audio_206.wav,3.294375
5,audio_161.wav,2.935208
6,audio_215.wav,2.962292
7,audio_213.wav,3.054792
8,audio_11.wav,3.408125
9,audio_155.wav,3.390833


In [32]:
# Check available speech-to-text libraries

import importlib.util

packages = [
    "whisper",
    "faster_whisper",
    "transformers",
    "torch"
]

print("Speech-to-text environment:")
for pkg in packages:
    available = importlib.util.find_spec(pkg) is not None
    print(f"{pkg:15} : {'AVAILABLE' if available else 'NOT AVAILABLE'}")


Speech-to-text environment:
whisper         : NOT AVAILABLE
faster_whisper  : NOT AVAILABLE
transformers    : AVAILABLE
torch           : AVAILABLE


In [37]:
# Check whether a Whisper model is already cached

import os

hf_cache = os.path.expanduser("~/.cache/huggingface/hub")

print("Hugging Face cache exists:", os.path.exists(hf_cache))

if os.path.exists(hf_cache):
    whisper_models = [
        name for name in os.listdir(hf_cache)
        if "whisper" in name.lower()
    ]
    print("Cached Whisper models:")
    if whisper_models:
        for name in whisper_models:
            print(" -", name)
    else:
        print("No cached Whisper model found.")
else:
    print("No Hugging Face cache found.")

Hugging Face cache exists: False
No Hugging Face cache found.


In [39]:
# Define fluency features + extract training features

def extract_fluency_features(filepath):
    y, sr = librosa.load(filepath, sr=16000, mono=True)

    features = {}

    # Energy
    rms = librosa.feature.rms(
        y=y,
        frame_length=2048,
        hop_length=512
    )[0]

    threshold = max(np.percentile(rms, 20), 1e-5)
    active_frames = rms > threshold

    features["active_speech_ratio"] = np.mean(active_frames)
    features["silence_ratio"] = 1 - features["active_speech_ratio"]

    # Zero crossing rate
    zcr = librosa.feature.zero_crossing_rate(
        y,
        frame_length=2048,
        hop_length=512
    )[0]

    features["zcr_median"] = np.median(zcr)
    features["zcr_min"] = np.min(zcr)
    features["zcr_max"] = np.max(zcr)

    # Energy statistics
    features["rms_median"] = np.median(rms)
    features["rms_min"] = np.min(rms)

    if len(rms) > 1:
        rms_diff = np.diff(rms)
        features["rms_change_mean"] = np.mean(np.abs(rms_diff))
        features["rms_change_std"] = np.std(rms_diff)
    else:
        features["rms_change_mean"] = 0
        features["rms_change_std"] = 0

    # Spectral flatness
    flatness = librosa.feature.spectral_flatness(
        y=y,
        n_fft=2048,
        hop_length=512
    )[0]

    features["flatness_mean"] = np.mean(flatness)
    features["flatness_std"] = np.std(flatness)

    # Spectral flux
    stft = np.abs(
        librosa.stft(
            y,
            n_fft=2048,
            hop_length=512
        )
    )

    if stft.shape[1] > 1:
        flux = np.sqrt(
            np.sum(np.diff(stft, axis=1) ** 2, axis=0)
        )
        features["spectral_flux_mean"] = np.mean(flux)
        features["spectral_flux_std"] = np.std(flux)
    else:
        features["spectral_flux_mean"] = 0
        features["spectral_flux_std"] = 0

    return features


# Extract training features
fluency_train_features = []

for filename in tqdm(
    train["filename"],
    desc="Processing training fluency features"
):
    filepath = os.path.join(TRAIN_DIR, filename)

    try:
        features = extract_fluency_features(filepath)
        features["filename"] = filename
        fluency_train_features.append(features)

    except Exception as e:
        print(f"Error processing {filename}: {e}")

fluency_train_features = pd.DataFrame(fluency_train_features)

print("\nTraining fluency feature extraction complete!")
print("Feature matrix shape:", fluency_train_features.shape)

display(fluency_train_features.head())

Processing training fluency features:   0%|          | 0/769 [00:00<?, ?it/s]


Training fluency feature extraction complete!
Feature matrix shape: (769, 14)


,active_speech_ratio,silence_ratio,zcr_median,zcr_min,zcr_max,rms_median,rms_min,rms_change_mean,rms_change_std,flatness_mean,flatness_std,spectral_flux_mean,spectral_flux_std,filename
0,0.799577,0.200423,0.072754,0.019531,0.335938,0.014546,0.000104,0.006581,0.011750,0.002581,0.004018,13.540026,16.972040,audio_192.wav
1,0.799787,0.200213,0.092285,0.021973,0.796387,0.064333,0.000282,0.013250,0.020798,0.023299,0.047190,36.051685,30.274675,audio_436.wav
2,0.799787,0.200213,0.098145,0.019043,0.699707,0.043020,0.000957,0.012913,0.020685,0.025846,0.046309,30.786633,30.777239,audio_447.wav
3,0.799858,0.200142,0.047852,0.016602,0.350098,0.010709,0.000037,0.002630,0.004235,0.002562,0.007569,6.403828,6.313538,audio_126.wav
4,0.799435,0.200565,0.132812,0.021484,0.725586,0.007482,0.000003,0.001798,0.002782,0.069948,0.125863,4.829741,4.879353,audio_61.wav


In [40]:
# Extract fluency features from all test audio

fluency_test_features = []

for filename in tqdm(
    test["filename"],
    desc="Processing test fluency features"
):
    filepath = os.path.join(TEST_DIR, filename)

    try:
        features = extract_fluency_features(filepath)
        features["filename"] = filename
        fluency_test_features.append(features)

    except Exception as e:
        print(f"Error processing {filename}: {e}")

fluency_test_features = pd.DataFrame(fluency_test_features)

print("\nTest fluency feature extraction complete!")
print("Feature matrix shape:", fluency_test_features.shape)

display(fluency_test_features.head())

Processing test fluency features:   0%|          | 0/216 [00:00<?, ?it/s]


Test fluency feature extraction complete!
Feature matrix shape: (216, 14)


,active_speech_ratio,silence_ratio,zcr_median,zcr_min,zcr_max,rms_median,rms_min,rms_change_mean,rms_change_std,flatness_mean,flatness_std,spectral_flux_mean,spectral_flux_std,filename
0,0.799858,0.200142,0.073242,0.024902,0.510742,0.015123,0.000266,0.004113,0.006322,0.022428,0.038125,9.795141,10.018433,audio_128.wav
1,0.799431,0.200569,0.171875,0.024902,0.726074,0.040865,0.000597,0.009905,0.017851,0.035010,0.046018,24.865063,26.564003,audio_156.wav
2,0.799574,0.200426,0.140137,0.035645,0.516113,0.040151,0.000504,0.012116,0.019554,0.034835,0.071029,27.738787,27.021559,audio_19.wav
3,0.799892,0.200108,0.151367,0.019043,0.744141,0.004074,0.000203,0.001041,0.001777,0.074434,0.089453,2.538591,2.683840,audio_108.wav
4,0.799716,0.200284,0.045898,0.000000,0.485352,0.062434,0.000000,0.009766,0.013847,0.014595,0.056984,27.571125,19.467386,audio_206.wav


In [41]:
# Combine enhanced acoustic + fluency features

fluency_cols = [
    col for col in fluency_train_features.columns
    if col != "filename"
]

enhanced_cols = [
    col for col in enhanced_train_features.columns
    if col != "filename"
]

X_combined = pd.concat(
    [
        enhanced_train_features[enhanced_cols].reset_index(drop=True),
        fluency_train_features[fluency_cols].reset_index(drop=True)
    ],
    axis=1
)

y = train["label"].values

print("Enhanced acoustic features:", len(enhanced_cols))
print("Fluency features:", len(fluency_cols))
print("Combined features:", X_combined.shape[1])

combined_model = make_pipeline(
    SimpleImputer(strategy="median"),
    ExtraTreesRegressor(
        n_estimators=400,
        random_state=42,
        n_jobs=-1,
        max_features=0.8,
        min_samples_leaf=2
    )
)

scores = cross_val_score(
    combined_model,
    X_combined,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=1
)

combined_rmse = -scores

print("\nCombined Model CV Results")
print("-------------------------")

for i, score in enumerate(combined_rmse, 1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMean RMSE: {combined_rmse.mean():.4f}")
print(f"Std RMSE : {combined_rmse.std():.4f}")

print("\nPrevious enhanced RMSE: 0.7437")

Enhanced acoustic features: 71
Fluency features: 13
Combined features: 84

Combined Model CV Results
-------------------------
Fold 1: 0.6814
Fold 2: 0.7092
Fold 3: 0.7812
Fold 4: 0.7275
Fold 5: 0.8023

Mean RMSE: 0.7403
Std RMSE : 0.0450

Previous enhanced RMSE: 0.7437


In [42]:
# Train combined acoustic + fluency model on all training data

X_test_combined = pd.concat(
    [
        enhanced_test_features[enhanced_cols].reset_index(drop=True),
        fluency_test_features[fluency_cols].reset_index(drop=True)
    ],
    axis=1
)

combined_model.fit(
    X_combined,
    y
)

combined_predictions = combined_model.predict(X_test_combined)

combined_predictions = np.clip(
    combined_predictions,
    0,
    5
)

print("Combined model trained successfully!")
print("Features:", X_combined.shape[1])
print("Predictions:", len(combined_predictions))
print("Minimum:", combined_predictions.min())
print("Maximum:", combined_predictions.max())
print("Mean:", combined_predictions.mean())

Combined model trained successfully!
Features: 84
Predictions: 216
Minimum: 2.4612499999999993
Maximum: 4.342083333333336
Mean: 3.245757137345679


In [43]:
# Create Submission #3 using acoustic + fluency features

submission_combined = pd.DataFrame({
    "filename": test["filename"],
    "label": combined_predictions
})

combined_submission_path = "/kaggle/working/submission_combined.csv"

submission_combined.to_csv(
    combined_submission_path,
    index=False
)

print("Combined submission created!")
print("Shape:", submission_combined.shape)
print("Saved to:", combined_submission_path)

print("\nValidation:")
print("Columns:", submission_combined.columns.tolist())
print("Missing values:")
print(submission_combined.isnull().sum())
print("Duplicate filenames:",
      submission_combined["filename"].duplicated().sum())

print(
    "Filename order matches test:",
    submission_combined["filename"].equals(test["filename"])
)

display(submission_combined.head(10))

Combined submission created!
Shape: (216, 2)
Saved to: /kaggle/working/submission_combined.csv

Validation:
Columns: ['filename', 'label']
Missing values:
filename    0
label       0
dtype: int64
Duplicate filenames: 0
Filename order matches test: True


,filename,label
0,audio_128.wav,3.280000
1,audio_156.wav,3.986667
2,audio_19.wav,3.632292
3,audio_108.wav,2.571250
4,audio_206.wav,3.268750
5,audio_161.wav,2.918333
6,audio_215.wav,3.000000
7,audio_213.wav,3.124167
8,audio_11.wav,3.506667
9,audio_155.wav,3.336875
